In [ ]:
"""
Baseline ResNet-50 for Thoracic Disease Classification
Based on: El-Fiky et al. (2021) - Multi-Label Transfer Learning

This notebook implements the baseline ResNet-50 model using transfer learning
methodology from the paper to achieve comparable performance (AUC: 0.911, F1: 0.66)
"""
import numpy as np
import pandas as pd
import os
import glob
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50V2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, hamming_loss, roc_curve
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

In [ ]:
# ============================================================================
# CONFIGURATION - Following El-Fiky et al. (2021) Paper
# ============================================================================

# Dataset configuration
IMAGE_SIZE = 224  # Paper uses 224x224
BATCH_SIZE = 32   # Paper uses batch size 32
EPOCHS = 60       # Paper uses 60 epochs
LEARNING_RATE = 0.0005  # Paper uses 0.001

print(f"\nConfiguration:")
print(f"  Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning Rate: {LEARNING_RATE}")

In [ ]:
# Cell 2: Dataset Configuration and Loading
BASE_DIR = "./dataset_pneumothorax/"
CSV_PATH = os.path.join(BASE_DIR, "balanced_dataset.csv")
IMAGE_DIR = os.path.join(BASE_DIR, "images")

print(f"Loading balanced Pneumothorax dataset...")

# Load the balanced dataset CSV
try:
    df = pd.read_csv(CSV_PATH)
    print(f"✓ Loaded CSV with {len(df):,} total images")
except Exception as e:
    print(f"✗ Error reading CSV {CSV_PATH}: {e}")
    exit(1)

# Display CSV structure
print(f"\n=== Dataset Structure ===")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows:")
print(df.head(3))

# Use Binary_Label column (already created by dataset.py)
if 'Binary_Label' in df.columns:
    df['binary_label'] = df['Binary_Label']
elif 'Pneumothorax' in df.columns:
    df['binary_label'] = df['Pneumothorax'].astype(int)
else:
    print("✗ Could not find Pneumothorax labels")
    exit(1)

# Create full image paths
df['path'] = df['Image_Index'].apply(lambda x: os.path.join(IMAGE_DIR, x))

# Verify images exist
df['exists'] = df['path'].apply(os.path.exists)
missing_count = (~df['exists']).sum()

if missing_count > 0:
    print(f"\n⚠ Warning: {missing_count} images not found")
    df = df[df['exists']].copy()

df = df.drop(columns=['exists'])
balanced_df = df.copy()

print(f"\n=== Final Dataset Summary ===")
print(f"Total images: {len(df):,}")
print(f"Pneumothorax (Positive): {df['binary_label'].sum():,} ({df['binary_label'].sum()/len(df)*100:.1f}%)")
print(f"No Pneumothorax (Negative): {(len(df) - df['binary_label'].sum()):,} ({(len(df) - df['binary_label'].sum())/len(df)*100:.1f}%)")
print(f"Balance ratio: 1:1")
print(f"\nDataset is ready for training!")

In [ ]:
# Cell 4: Dataset Splitting with Stratification
# Path already exists from Cell 2, just use balanced_df directly
print(f"Dataset ready for splitting: {len(balanced_df)} samples")

# Stratified train-validation-test split (80/10/10)
train_df, temp_df = train_test_split(
    balanced_df,
    test_size=0.2,
    random_state=42,
    stratify=balanced_df['binary_label']
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df['binary_label']
)

print(f"\n=== Dataset Splits (Stratified) ===")
print(f"Training set: {len(train_df):,} samples")
print(f"  Positive: {train_df['binary_label'].sum():,} ({train_df['binary_label'].sum()/len(train_df)*100:.1f}%)")
print(f"  Negative: {len(train_df) - train_df['binary_label'].sum():,} ({(len(train_df) - train_df['binary_label'].sum())/len(train_df)*100:.1f}%)")

print(f"\nValidation set: {len(val_df):,} samples")
print(f"  Positive: {val_df['binary_label'].sum():,} ({val_df['binary_label'].sum()/len(val_df)*100:.1f}%)")
print(f"  Negative: {len(val_df) - val_df['binary_label'].sum():,} ({(len(val_df) - val_df['binary_label'].sum())/len(val_df)*100:.1f}%)")

print(f"\nTest set: {len(test_df):,} samples")
print(f"  Positive: {test_df['binary_label'].sum():,} ({test_df['binary_label'].sum()/len(test_df)*100:.1f}%)")
print(f"  Negative: {len(test_df) - test_df['binary_label'].sum():,} ({(len(test_df) - test_df['binary_label'].sum())/len(test_df)*100:.1f}%)")

# Save splits
os.makedirs('./output', exist_ok=True)
test_df.to_csv('./output/pneumothorax_test_set.csv', index=False)
print("\n✓ Test set saved for final evaluation")

In [ ]:
# ============================================================================
# TF.DATA PIPELINE - BINARY CLASSIFICATION
# ============================================================================

def create_tf_dataset(df, batch_size, shuffle=True, augment=True):
    """Enhanced tf.data pipeline with better augmentation"""
    
    def load_and_preprocess(image_path, label):
        # Load image
        img = tf.io.read_file(image_path)
        img = tf.image.decode_png(img, channels=3)
        img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
        
        # ENHANCED AUGMENTATION for medical images
        if augment:
            img = tf.image.random_flip_left_right(img)
            img = tf.image.random_flip_up_down(img)  # Medical images can be flipped vertically
            
            # Random rotation (small angles for medical images)
            img = tf.image.rot90(img, k=tf.random.uniform([], 0, 4, dtype=tf.int32))
            
            # Random brightness and contrast (subtle for X-rays)
            img = tf.image.random_brightness(img, max_delta=0.1)
            img = tf.image.random_contrast(img, lower=0.9, upper=1.1)
            
            # Add slight zoom by random crop and resize
            if tf.random.uniform([]) > 0.5:
                crop_size = tf.random.uniform([], 0.85, 1.0)
                h, w = IMAGE_SIZE, IMAGE_SIZE
                crop_h = tf.cast(h * crop_size, tf.int32)
                crop_w = tf.cast(w * crop_size, tf.int32)
                img = tf.image.random_crop(img, [crop_h, crop_w, 3])
                img = tf.image.resize(img, [IMAGE_SIZE, IMAGE_SIZE])
        
        # Normalize using ResNet50V2 preprocessing
        img = tf.keras.applications.resnet_v2.preprocess_input(img)
        return img, label
    
    # Create dataset
    image_paths = df['path'].values
    labels = df['binary_label'].values.astype(np.float32)
    
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    
    if shuffle:
        dataset = dataset.shuffle(buffer_size=2000, seed=42)  # Increased buffer
    
    dataset = dataset.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    
    return dataset

# Create datasets
print("Creating tf.data pipelines for binary classification...")
train_dataset = create_tf_dataset(train_df, BATCH_SIZE, shuffle=True, augment=True)
val_dataset = create_tf_dataset(val_df, BATCH_SIZE, shuffle=False, augment=False)
test_dataset = create_tf_dataset(test_df, BATCH_SIZE, shuffle=False, augment=False)

print(f"✅ tf.data pipelines created:")
print(f"  Training batches: {len(train_dataset)}")
print(f"  Validation batches: {len(val_dataset)}")
print(f"  Test batches: {len(test_dataset)}")

# Verify dataset structure
for images, labels in train_dataset.take(1):
    print(f"\nDataset verification:")
    print(f"  Image batch shape: {images.shape}")
    print(f"  Label batch shape: {labels.shape}")
    print(f"  Label range: {tf.reduce_min(labels).numpy():.1f} to {tf.reduce_max(labels).numpy():.1f}")
    print(f"  Sample labels: {labels[:5].numpy()}")

In [ ]:
# ============================================================================
# MODEL ARCHITECTURE - ResNet-50 Transfer Learning (Paper Methodology)
# ============================================================================

def create_baseline_resnet50(
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    dropout_rate=0.3  # Add dropout for regularization
):
    """
    Enhanced ResNet-50 with better regularization
    
    Key improvements:
    - Fixed the fine_tune bug
    - Added dropout for regularization
    - Better layer configuration
    - Proper BatchNormalization
    """
    
    # Load ResNet-50 pre-trained on ImageNet
    base_model = ResNet50V2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape,
        pooling='avg'
    )
    
    # Initially freeze base model, then unfreeze after a few epochs
    # This is more stable than training everything from start
    base_model.trainable = False
    
    # Build model
    inputs = layers.Input(shape=input_shape)
    x = base_model(inputs, training=False)
    
    # Add regularization layers
    x = layers.Dropout(dropout_rate, name='dropout')(x)
    
    # Output layer
    outputs = layers.Dense(
        1, 
        activation='sigmoid',
        kernel_regularizer=tf.keras.regularizers.l2(0.01),  # L2 regularization
        name='predictions'
    )(x)
    
    model = keras.Model(inputs, outputs, name='PneumothoraxModel')
    
    return model, base_model

print("\nCreating Enhanced ResNet-50 model...")
model, base_model = create_baseline_resnet50(dropout_rate=0.3)

print(f"\nModel Architecture:")
print(f"  Total parameters: {model.count_params():,}")
trainable_params = sum([tf.size(v).numpy() for v in model.trainable_variables])
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Base model frozen: {not base_model.trainable}")

In [ ]:
# Calculate class weights to handle any residual imbalance
from sklearn.utils.class_weight import compute_class_weight

class_weights_array = compute_class_weight(
    'balanced',
    classes=np.unique(train_df['binary_label']),
    y=train_df['binary_label']
)
class_weights = dict(enumerate(class_weights_array))

print(f"\nClass weights: {class_weights}")
print(f"  Negative (0): {class_weights[0]:.3f}")
print(f"  Positive (1): {class_weights[1]:.3f}")

In [ ]:
# ============================================================================
# MODEL COMPILATION - Following Paper Configuration
# ============================================================================

model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='accuracy')
    ]
)

print("\nModel compiled with:")
print(f"  Optimizer: Adam (lr={LEARNING_RATE})")
print(f"  Loss: binary_crossentropy")
print(f"  Class weights: Applied")

In [ ]:
# ============================================================================
# CALLBACKS
# ============================================================================

# Create output directory
os.makedirs('./output', exist_ok=True)

callbacks = [
    # Model checkpoint - save best model based on validation AUC
    ModelCheckpoint(
        filepath='./output/baseline_resnet50_best.keras',
        monitor='val_auc',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    
    # Early stopping with more patience for frozen->unfrozen transition
    EarlyStopping(
        monitor='val_auc',
        patience=15,  # Increased patience
        mode='max',
        restore_best_weights=True,
        verbose=1,
        min_delta=0.001  # Minimum improvement threshold
    ),
    
    # Learning rate reduction - more aggressive
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,  # Less aggressive reduction
        patience=5,  # Increased patience
        min_lr=1e-7,
        verbose=1,
        min_delta=0.0001
    )
]

In [ ]:
# ============================================================================
# TRAINING
# ============================================================================

print("\n" + "="*80)
print("ENHANCED TWO-STAGE TRAINING")
print("="*80)
print("Stage 1: Train with frozen base (transfer learning)")
print("Stage 2: Fine-tune entire network")
print("="*80 + "\n")

# STAGE 1: Train with frozen base model (10 epochs)
print("\n" + "="*80)
print("STAGE 1: Training with Frozen Base Model")
print("="*80)

history_stage1 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=10,  # Initial epochs with frozen base
    callbacks=callbacks,
    class_weight=class_weights,  # Apply class weights
    verbose=1
)

# STAGE 2: Unfreeze and fine-tune entire network
print("\n" + "="*80)
print("STAGE 2: Fine-tuning Entire Network")
print("="*80)

# Unfreeze the base model
base_model.trainable = True

# Recompile with lower learning rate for fine-tuning
model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE * 0.1),  # 10x lower LR
    loss='binary_crossentropy',
    metrics=[
        'binary_accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.BinaryAccuracy(threshold=0.5, name='accuracy')
    ]
)

print(f"\nFine-tuning with learning rate: {LEARNING_RATE * 0.1}")
print(f"Trainable parameters: {sum([tf.size(v).numpy() for v in model.trainable_variables]):,}")

# Continue training
history_stage2 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=40,  # Additional epochs for fine-tuning
    initial_epoch=10,  # Continue from where stage 1 left off
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1
)

# Combine histories
history = history_stage1
for key in history_stage2.history.keys():
    history.history[key].extend(history_stage2.history[key])

# Save final model
model.save('./output/baseline_resnet50_final.keras')
print("\n✓ Training complete!")
print("Models saved:")
print("  - Best model: ./output/baseline_resnet50_best.keras")
print("  - Final model: ./output/baseline_resnet50_final.keras")

In [ ]:
# ============================================================================
# TRAINING HISTORY VISUALIZATION
# ============================================================================

def plot_training_history(history):
    """Plot training metrics"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Loss
    axes[0, 0].plot(history.history['loss'], label='Training Loss')
    axes[0, 0].plot(history.history['val_loss'], label='Validation Loss')
    axes[0, 0].set_title('Model Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # AUC
    axes[0, 1].plot(history.history['auc'], label='Training AUC')
    axes[0, 1].plot(history.history['val_auc'], label='Validation AUC')
    axes[0, 1].set_title('Model AUC')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('AUC')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Precision
    axes[1, 0].plot(history.history['precision'], label='Training Precision')
    axes[1, 0].plot(history.history['val_precision'], label='Validation Precision')
    axes[1, 0].set_title('Model Precision')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Precision')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Recall
    axes[1, 1].plot(history.history['recall'], label='Training Recall')
    axes[1, 1].plot(history.history['val_recall'], label='Validation Recall')
    axes[1, 1].set_title('Model Recall')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('./output/baseline_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_training_history(history)

In [ ]:
# ============================================================================
# COMPREHENSIVE EVALUATION - BINARY CLASSIFICATION
# ============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE EVALUATION")
print("="*80)

# Generate predictions on validation dataset
print("Generating predictions on validation set...")
val_predictions = model.predict(val_dataset, verbose=1)

# Get true labels from validation dataframe - FIXED
y_true = val_df['binary_label'].values  # Use binary_label column
y_pred_probs = val_predictions.flatten()  # Flatten to 1D array
y_pred = (y_pred_probs > 0.5).astype(int)  # Convert probabilities to binary predictions

print(f"\nPredictions shape: {y_pred_probs.shape}")
print(f"True labels shape: {y_true.shape}")
print(f"Unique predictions: {np.unique(y_pred)}")
print(f"Unique true labels: {np.unique(y_true)}")

# Calculate metrics
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# AUC Score
auc_score = roc_auc_score(y_true, y_pred_probs)
print(f"\n📊 VALIDATION METRICS:")
print(f"  AUC Score: {auc_score:.4f}")

# Classification Report
print(f"\n📋 Classification Report:")
print(classification_report(y_true, y_pred, target_names=['No Pneumothorax', 'Pneumothorax']))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print(f"\n🔢 Confusion Matrix:")
print(f"  True Negatives: {cm[0,0]}")
print(f"  False Positives: {cm[0,1]}")
print(f"  False Negatives: {cm[1,0]}")
print(f"  True Positives: {cm[1,1]}")

# Calculate additional metrics
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)  # Recall
specificity = tn / (tn + fp)
precision = tp / (tp + fp)
f1_score = 2 * (precision * sensitivity) / (precision + sensitivity)

print(f"\n📈 Detailed Metrics:")
print(f"  Sensitivity (Recall): {sensitivity:.4f}")
print(f"  Specificity: {specificity:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  F1-Score: {f1_score:.4f}")
print(f"  Accuracy: {(tp + tn) / (tp + tn + fp + fn):.4f}")

In [ ]:
# ============================================================================
# VISUALIZE PREDICTIONS - FIXED
# ============================================================================

def visualize_predictions(test_df, y_pred_probs, n_samples=12):
    """Visualize sample predictions"""
    # Get sample indices
    indices = np.random.choice(len(test_df), n_samples, replace=False)
    
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.ravel()
    
    for idx, img_idx in enumerate(indices):
        img_path = test_df.iloc[img_idx]['path']
        true_label = test_df.iloc[img_idx]['binary_label']  # FIXED: Use 'binary_label'
        pred_prob = y_pred_probs[img_idx]  # FIXED: Remove [0] since it's already 1D
        pred_label = 1 if pred_prob > 0.5 else 0
        
        # Load and display image
        img = keras.preprocessing.image.load_img(img_path, target_size=(224,224))
        img_array = keras.preprocessing.image.img_to_array(img)
        
        axes[idx].imshow(img_array.astype('uint8'))
        
        # Create title with prediction info
        true_class = "Pneumothorax" if true_label == 1 else "Negative"
        pred_class = "Pneumothorax" if pred_label == 1 else "Negative"
        color = 'green' if true_label == pred_label else 'red'
        
        title = f"True: {true_class}\nPred: {pred_class} ({pred_prob:.3f})"
        axes[idx].set_title(title, color=color, fontweight='bold')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig('./output/sample_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

# Get predictions on test set first
print("Generating predictions on test set...")
test_predictions = model.predict(test_dataset, verbose=1)
test_pred_probs = test_predictions.flatten()  # Already 1D

print(f"Test predictions shape: {test_pred_probs.shape}")
print(f"Test dataframe shape: {test_df.shape}")

# Visualize predictions
visualize_predictions(test_df.reset_index(drop=True), test_pred_probs)

In [ ]:
# Load best model

# Evaluate on val and test
val_metrics = model.evaluate(val_dataset)
test_metrics = model.evaluate(test_dataset)

print(f"Val AUC: {val_metrics}")
print(f"Test AUC: {test_metrics}")